# Assignment 2B - RAG Pipeline

**Domain:** Type 2 diabetes management. **Total:** 20 marks.

| Part | Scope | Marks |
| --- | --- | --- |
| A | Dense search and chunk size | 5 |
| B | Dense, BM25, and hybrid search | 4 |
| C | Cross-encoder reranking | 4 |
| D | Grounded generation and LLM evaluation | 7 |

| Role | Model / method |
| --- | --- |
| Generator | Assignment 1 Mistral-7B-v0.1 + CPT + QLoRA adapter |
| Embeddings | BAAI/bge-base-en-v1.5 (512-token input limit) |
| Dense index | FAISS IndexFlatIP with normalized embeddings |
| Keyword / hybrid | BM25 / reciprocal rank fusion |
| Reranker | cross-encoder/ms-marco-MiniLM-L-6-v2 |
| Judge | Qwen/Qwen2.5-7B-Instruct |

## Setup

Run on the CUDA lab server with the Assignment 1 corpus, evaluation dataset, CPT checkpoint, and adapter available.

1. Configure the input paths and confirm `EVAL_SPLIT_METHOD` and `EVAL_SPLIT_EVIDENCE`.
2. Review `SPAN_REVIEW` on the first run; approved queries and spans are saved for reuse.
3. After generation, complete `SPOT_REVIEW` and `FAILURE_REVIEW` using the displayed evidence.
4. Keep all outputs and export the completed notebook to HTML.

The notebook pauses at unconfirmed inputs or annotations rather than scoring unverified data.


In [ ]:
import importlib.util
import os
from types import SimpleNamespace


def detect_env():
    """Return "kubeflow", "colab" or "local". Kubeflow is checked first."""
    if ("NB_PREFIX" in os.environ or "KUBERNETES_SERVICE_HOST" in os.environ
            or os.path.isdir("/home/jovyan/data")):
        return "kubeflow"
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "colab"
    except ModuleNotFoundError:
        pass
    return "local"


def setup():
    """Pick the storage folder, set the Hugging Face cache, and read the device and dtype."""
    env = detect_env()
    candidates = {"colab": ["/content"], "kubeflow": ["/home/jovyan/assessment", "/home/jovyan/data"],
                  "local": [os.path.expanduser("~/lab-data"), os.getcwd()]}[env]
    root = os.getcwd()
    # Test write access before choosing where large downloaded model files will live.
    for path in candidates:
        try:
            os.makedirs(path, exist_ok=True)
            probe = os.path.join(path, ".write_test")
            open(probe, "w").close()
            os.remove(probe)
            root = path
            break
        except Exception:
            continue

    # Set the cache location before importing Hugging Face model libraries.
    cache_root = os.path.join(root, "hf-cache")
    os.environ["HF_HOME"] = cache_root
    os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

    device, dtype, gpu_name, gpu_mem = "cpu", "float32", None, 0.0
    try:
        import torch
        if torch.cuda.is_available():
            device = "cuda"
            gpu_name = torch.cuda.get_device_name(0)
            gpu_mem = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
            # Use BF16 only when the GPU supports it; otherwise use FP16.
            dtype = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
    except Exception:
        pass

    state = SimpleNamespace(ENV=env, ROOT=root, CACHE_ROOT=cache_root, DEVICE=device,
                            DTYPE=dtype, IS_GPU=(device == "cuda"), GPU_NAME=gpu_name, GPU_MEM_GB=gpu_mem)
    print(f"env={env}  device={device}  dtype={dtype}  gpu={gpu_name or 'none'} ({gpu_mem} GB)")
    print(f"storage root : {root}")
    print(f"HF cache     : {cache_root}")
    return state


L = setup()

In [ ]:
import json
import random
import re
from pathlib import Path

SEED = 42
random.seed(SEED)

# Reuse the cleaned corpus, CPT checkpoint, and trained adapter from Assignment 1.
ASS1 = Path("../Ass-1").resolve()
CORPUS_DIR = ASS1 / "domain_corpus"
DATASET_PATH = ASS1 / "instruction_dataset.jsonl"
CPT_DIR = ASS1 / "persistent" / "cpt_ckpt"
ADAPTER_DIR = ASS1 / "persistent" / "adapter"
# Confirm the real Assignment 1 split and record its evidence before evaluating.
EVAL_SPLIT_METHOD = "last_80"
EVAL_SPLIT_EVIDENCE = ""
EVAL_PATH = ASS1 / "instruction_eval.jsonl"

print(f"Assignment 1 folder: {ASS1}")
print(f"CPT checkpoint: {CPT_DIR}")
print(f"QLoRA adapter: {ADAPTER_DIR}")
print(f"HF cache: {L.CACHE_ROOT}")

DATA_DIR, OUT_DIR = Path("data"), Path("outputs")
for folder in (DATA_DIR, OUT_DIR):
    folder.mkdir(exist_ok=True)


def count_rows(path):
    with open(path, encoding="utf-8") as handle:
        return sum(1 for line in handle if line.strip())


# Fail early on missing lab inputs rather than reaching model loading with invalid paths.
checks = {
    "cleaned corpus .txt files": CORPUS_DIR.is_dir() and bool(list(CORPUS_DIR.glob("*.txt"))),
    "instruction dataset": DATASET_PATH.is_file(),
    "CPT checkpoint": CPT_DIR.is_dir(),
    "QLoRA adapter": ADAPTER_DIR.is_dir(),
}
for label, present in checks.items():
    print(f"{'OK' if present else 'MISSING':8} {label}")
if not all(checks.values()):
    raise FileNotFoundError("Restore the Assignment 1 inputs on the lab server before continuing.")

### Dependencies and Utilities

Install missing packages, record versions, and define shared display and timing helpers.


In [ ]:
import importlib.metadata
import importlib.util
import subprocess
import sys

PACKAGES = {
    "numpy": "numpy",
    "torch": "torch",
    "packaging": "packaging",
    "sentence-transformers": "sentence_transformers",
    "faiss-cpu": "faiss",
    "rank-bm25": "rank_bm25",
    "transformers": "transformers",
    "peft": "peft",
    "bitsandbytes": "bitsandbytes",
    "accelerate": "accelerate",
    "pandas": "pandas",
}
missing = [package for package, module in PACKAGES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)

import numpy as np
import torch

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
L.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
L.IS_GPU = L.DEVICE == "cuda"
L.DTYPE = ("bfloat16" if torch.cuda.is_bf16_supported() else "float16") if L.IS_GPU else "float32"
versions = {package: importlib.metadata.version(package) for package in PACKAGES}
(OUT_DIR / "package_versions.json").write_text(json.dumps(versions, indent=2), encoding="utf-8")
for package, version in versions.items():
    print(f"{package}: {version}")
print(f"Device: {L.DEVICE}; compute dtype: {L.DTYPE}")


In [ ]:
def print_table(headers, rows, title=None, align=None, save_as=None):
    """Print an aligned text table.

    headers   : column titles
    rows      : list of tuples. Each cell is already a string, so the caller controls
                separators and the dash used for values that do not apply.
    title     : optional heading printed above the table
    align     : per-column "l" or "r". The default is label on the left and numbers on the right.
    save_as   : optional file name. The table is also saved as outputs/<save_as>.csv.
    """
    align = align or ["l"] + ["r"] * (len(headers) - 1)
    body = list(rows)
    widths = [max(len(headers[c]), max(len(r[c]) for r in body)) for c in range(len(headers))]
    gap = "   "
    width = sum(widths) + len(gap) * (len(widths) - 1)

    def line(cells):
        """Format one table row: pad each cell to its column width."""
        return gap.join(c.ljust(w) if a == "l" else c.rjust(w)
                        for c, w, a in zip(cells, widths, align))

    if title:
        print("=" * width)
        print(title)
    print("=" * width)
    print(line(headers))
    print("-" * width)
    for r in rows:
        print(line(r))
    print("=" * width)

    if save_as:
        import csv
        with open(OUT_DIR / f"{save_as}.csv", "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(headers)
            writer.writerows(rows)


def show_md_table(headers, rows, title=None, save_as=None):
    """Show a table as Markdown, so long text wraps in the notebook.

    Used for tables with long text, such as queries, spans and judge reasons.
    save_as also writes outputs/<save_as>.csv.
    """
    from IPython.display import Markdown, display

    def cell(text):
        """Keep a table cell on one line, and replace "|" so it does not start a new column."""
        return " ".join(str(text).split()).replace("|", "/")

    md = f"**{title}**\n\n" if title else ""
    md += "| " + " | ".join(headers) + " |\n| " + " | ".join("---" for _ in headers) + " |\n"
    md += "\n".join("| " + " | ".join(cell(c) for c in r) + " |" for r in rows)
    display(Markdown(md))
    if save_as:
        import csv
        with open(OUT_DIR / f"{save_as}.csv", "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(headers)
            writer.writerows(rows)


In [ ]:
import time
from contextlib import contextmanager


@contextmanager
def working(label, folder=None, expected_gb=None):
    started = time.perf_counter()
    print(f"{label}...")
    try:
        yield
    except Exception:
        print(f"{label}: failed after {time.perf_counter() - started:.1f}s")
        raise
    else:
        print(f"{label}: completed in {time.perf_counter() - started:.1f}s")

## Evaluation Queries and Answer Spans

Use ten queries and unchanged expected answers from Assignment 1. Select `last_80` only if the original split used the final 80 rows without shuffling; otherwise select `file` and set `EVAL_PATH`. Record the split source in `EVAL_SPLIT_EVIDENCE`.

Word overlap proposes candidate spans. Human review must confirm that each span answers its question. Approved queries, references, spans, and input hashes are frozen for later runs. Candidate selection favors overlap and document diversity, so conclusions apply only to this small benchmark.


In [ ]:
import hashlib

if EVAL_SPLIT_METHOD not in {"last_80", "file"}:
    raise ValueError("EVAL_SPLIT_METHOD must be 'last_80' or 'file'.")
if not EVAL_SPLIT_EVIDENCE.strip():
    raise ValueError("Set EVAL_SPLIT_EVIDENCE to the Assignment 1 split cell/artifact before selecting queries.")
split_path = DATASET_PATH if EVAL_SPLIT_METHOD == "last_80" else EVAL_PATH
with open(split_path, encoding="utf-8") as handle:
    split_rows = [json.loads(line) for line in handle if line.strip()]
if EVAL_SPLIT_METHOD == "last_80":
    if len(split_rows) != 400:
        raise ValueError("The documented last-80 split expects 400 dataset rows; verify Assignment 1.")
    eval_rows = split_rows[-80:]
else:
    eval_rows = split_rows
if len(eval_rows) < 10:
    raise ValueError("At least ten evaluation queries are required.")
for row in eval_rows:
    if not all(isinstance(row.get(field), str) and row[field].strip()
               for field in ("instruction", "response")):
        raise ValueError("Evaluation rows need nonempty instruction and response strings.")

corpus_files = sorted(CORPUS_DIR.glob("*.txt"))
INPUT_HASHES = {"split": hashlib.sha256(split_path.read_bytes()).hexdigest(),
                "corpus": {path.name: hashlib.sha256(path.read_bytes()).hexdigest()
                           for path in corpus_files}}
MANIFEST_PATH = DATA_DIR / "answer_spans.json"

def norm(text):
    return " ".join(text.lower().split())

CORPUS_TEXT = {path.name: path.read_text(encoding="utf-8") for path in corpus_files}
CORPUS_NORM = norm(" ".join(CORPUS_TEXT.values()))
STOP = set("the a an of and or to in on for with by is are was were be as at from that this it its which".split())

def content_words(text):
    return set(re.findall(r"[a-z0-9]+", text.lower())) - STOP

if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    if not isinstance(manifest, dict) or manifest.get("input_hashes") != INPUT_HASHES:
        raise ValueError("The saved benchmark is legacy or its inputs changed. Review it explicitly before replacing it.")
    if (manifest.get("split_method") != EVAL_SPLIT_METHOD
            or manifest.get("split_evidence") != EVAL_SPLIT_EVIDENCE):
        raise ValueError("Split provenance changed; review the saved benchmark before continuing.")
    QUERIES = manifest["queries"]
    print("Loaded the frozen, reviewed benchmark.")
else:
    sentences = [(name, sentence) for name, text in CORPUS_TEXT.items()
                 for sentence in re.split(r"(?<=[.!?])\s+", " ".join(text.split()))
                 if 8 <= len(sentence.split()) <= 40]
    if not sentences:
        raise ValueError("No candidate sentences found; annotate answer spans manually.")
    candidates = []
    for row in eval_rows:
        wanted = content_words(row["response"])
        source, span = max(sentences, key=lambda pair: len(wanted & content_words(pair[1])) /
                          max(len(content_words(pair[1])), 1) ** 0.5)
        if CORPUS_NORM.count(norm(span)) == 1:
            candidates.append({"query": row["instruction"], "expected_answer": row["response"],
                               "source_file": source, "span": span,
                               "overlap": len(wanted & content_words(span))})
    selected, seen_files, seen_queries = [], set(), set()
    for candidate in sorted(candidates, key=lambda item: -item["overlap"]):
        if candidate["source_file"] not in seen_files and candidate["query"] not in seen_queries:
            selected.append(candidate)
            seen_files.add(candidate["source_file"])
            seen_queries.add(candidate["query"])
        if len(selected) == 10:
            break
    if len(selected) != 10:
        raise ValueError("Could not propose ten distinct queries; select and annotate ten evaluation queries manually.")
    QUERIES = [{"qid": f"Q{number}", **{key: value for key, value in item.items() if key != "overlap"}}
               for number, item in enumerate(selected, 1)]
print(f"Evaluation rows: {len(eval_rows)}; selected queries: {len(QUERIES)}")


### Review the Spans

Read the candidate table, then fill `SPAN_REVIEW` using each exact question as its key. Each entry needs `approved: True`, `source_file`, and a verbatim `span` that answers the question. Keep the Assignment 1 reference unchanged. All ten annotations must be approved before retrieval scoring.


In [ ]:
# Key entries by exact question text; approve only after reading the source and reference.
SPAN_REVIEW = {}

if not MANIFEST_PATH.exists():
    show_md_table(["Query", "Expected answer", "Candidate source", "Candidate span"],
                  [(item["query"], item["expected_answer"], item["source_file"], item["span"])
                   for item in QUERIES], title="Candidates to review before retrieval scoring")
    pending = []
    for item in QUERIES:
        review = SPAN_REVIEW.get(item["query"], {})
        if review.get("approved") is not True:
            pending.append(item["qid"])
            continue
        item["source_file"] = review["source_file"]
        item["span"] = review["span"]
        item["human_reviewed"] = True
    # Candidate overlap is not proof that a span answers its question; require human approval.
    if pending:
        raise ValueError(f"Review SPAN_REVIEW for {', '.join(pending)}, then rerun this cell.")

# Preserve the original evaluation references and verify each span against its named document.
expected_pairs = {(row["instruction"], row["response"]) for row in eval_rows}
assert len(QUERIES) == 10
assert len({item["query"] for item in QUERIES}) == 10
assert [item["qid"] for item in QUERIES] == [f"Q{number}" for number in range(1, 11)]
for item in QUERIES:
    assert item.get("human_reviewed") is True, f"{item['qid']}: annotation is not reviewed"
    assert (item["query"], item["expected_answer"]) in expected_pairs, "Query/reference is not from the evaluation split"
    source = CORPUS_TEXT.get(item["source_file"], "")
    assert item["span"].strip() and norm(item["span"]) in norm(source), "Span is absent from its named source"
print("All ten query/reference pairs and human-reviewed source spans validated.")

In [ ]:
def is_hit(chunk_text, span):
    # Retrieval scoring uses a normalized literal span match, not a semantic similarity guess.
    return norm(span) in norm(chunk_text)

# Freeze reviewed annotations together with input hashes for reuse across all methods.
manifest = {"split_method": EVAL_SPLIT_METHOD, "split_evidence": EVAL_SPLIT_EVIDENCE,
            "input_hashes": INPUT_HASHES, "queries": QUERIES}
if not MANIFEST_PATH.exists():
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
show_md_table(["qid", "source file", "query", "expected answer", "answer span"],
              [(item["qid"], item["source_file"], item["query"], item["expected_answer"], item["span"])
               for item in QUERIES], title="Frozen evaluation benchmark", save_as="query_table")

## Part A - Dense Search and Chunk Size (5 marks)

Compare non-overlapping 100- and 400-token chunks using the embedding tokenizer. BGE's 512-token limit accommodates both sizes without embedding truncation. The PDF table says 500, but its instructions and checklist specify 400; this notebook follows 100/400.


In [ ]:
from sentence_transformers import SentenceTransformer
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()

EMBED_MODEL_ID = "BAAI/bge-base-en-v1.5"
EMBED_BATCH = 64

with working("Loading the embedding model"):
    embed_model = SentenceTransformer(EMBED_MODEL_ID, device=L.DEVICE)
embed_tok = embed_model.tokenizer
EMBED_DIM = embed_model.get_sentence_embedding_dimension()
print(f"model              : {EMBED_MODEL_ID}")
print(f"max_seq_length     : {embed_model.max_seq_length} tokens")
print(f"vector size        : {EMBED_DIM}")


### A1.1 Dense search function (1 mark)

One function embeds the chunks, builds a FAISS `IndexFlatIP` index and returns the top k chunks for each query. Embeddings are L2-normalised, so the inner product equals cosine similarity. The index is cached per chunk set, so Parts B and C reuse it.


In [ ]:
import time

import faiss

# Cache each chunk set so repeated searches do not rebuild its embeddings and index.
_INDEX_CACHE = {}


def dense_search(chunks, queries, k):
    """Embed the chunks (once per chunk set), build a FAISS IndexFlatIP, and return the top k chunks for each query.

    Returns one list per query. Each item is a dict with rank (from 1), chunk_id, score and text.
    """
    key = (id(chunks), len(chunks))
    if key not in _INDEX_CACHE:
        t0 = time.perf_counter()
        # Unit-length vectors make inner product equivalent to cosine similarity.
        vectors = embed_model.encode([c["text"] for c in chunks], batch_size=EMBED_BATCH,
                                     normalize_embeddings=True, convert_to_numpy=True).astype("float32")
        index = faiss.IndexFlatIP(vectors.shape[1])
        index.add(vectors)
        _INDEX_CACHE[key] = (chunks, index, time.perf_counter() - t0)
    _, index, _ = _INDEX_CACHE[key]

    # Queries must use the same normalization as the indexed chunks.
    query_vectors = embed_model.encode(queries, batch_size=EMBED_BATCH,
                                       normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, ids = index.search(query_vectors, k)
    results = []
    for qi in range(len(queries)):
        # FAISS uses -1 for unfilled slots when k exceeds the number of chunks.
        results.append([{"rank": r + 1, "chunk_id": chunks[j]["chunk_id"], "score": float(s), "text": chunks[j]["text"]}
                        for r, (s, j) in enumerate(zip(scores[qi], ids[qi])) if j >= 0])
    return results


def index_build_seconds(chunks):
    """Seconds spent embedding the chunks and adding them to the index, for this chunk set."""
    if (id(chunks), len(chunks)) not in _INDEX_CACHE:
        dense_search(chunks, [QUERIES[0]["query"]], 1)
    return _INDEX_CACHE[(id(chunks), len(chunks))][2]
print("dense_search() defined: embeds the chunks, builds a FAISS IndexFlatIP index, returns the top k.")

### A2.1 Chunk size test (2 marks)

Fixed-size chunks with no overlap, per document, counted with the embedding tokenizer. The text of each chunk is sliced from the original document with the tokenizer's offsets, so the generator sees the original wording.

| Column | How |
| --- | --- |
| Total chunks | number of chunks over the 14 documents |
| Hit rate@3 (%) | share of the 10 queries where one of the top 3 chunks contains the span |
| Tokens sent per query (top 3) | combined `n_tokens` of the top 3, averaged over the 10 queries |


In [ ]:
def chunk_corpus(size):
    """Split each .txt file into chunks of `size` tokens, with no overlap.

    Tokens are counted with the embedding tokenizer. Each chunk's text is cut from the original
    file, using the token offsets, so the wording and capitals are kept.
    """
    chunks = []
    for f in sorted(CORPUS_DIR.glob("*.txt")):
        text = f.read_text(encoding="utf-8")
        # Character offsets preserve the original wording rather than reconstructing decoded tokens.
        offsets = embed_tok(text, add_special_tokens=False, return_offsets_mapping=True)["offset_mapping"]
        # A full-size stride gives non-overlapping chunks; the final chunk may be shorter.
        for start in range(0, len(offsets), size):
            part = offsets[start:start + size]
            chunks.append({"chunk_id": f"{f.stem}#{start // size}", "doc": f.name,
                           "text": text[part[0][0]:part[-1][1]], "n_tokens": len(part)})
    return chunks
print("chunk_corpus() defined: fixed-size token chunks, no overlap.")

In [ ]:
# Compare the same ten queries at both required chunk sizes.
A2_SIZES = [100, 400]
A2_HIT = {}
A2_TOKENS = {}
chunk_sets = {}
a2_rows = []
for size in A2_SIZES:
    chunks_for_size = chunk_corpus(size)
    assert max(c["n_tokens"] for c in chunks_for_size) <= size
    chunk_sets[size] = chunks_for_size
    by_id = {c["chunk_id"]: c for c in chunks_for_size}
    tops = dense_search(chunks_for_size, [q["query"] for q in QUERIES], 3)
    # Count each query once if any of its top three chunks contains the annotated span.
    hits = sum(any(is_hit(r["text"], q["span"]) for r in top) for q, top in zip(QUERIES, tops))
    # Context cost is the combined chunk length, averaged across queries.
    tokens = sum(sum(by_id[r["chunk_id"]]["n_tokens"] for r in top) for top in tops) / len(QUERIES)
    A2_HIT[size] = 100 * hits / len(QUERIES)
    A2_TOKENS[size] = tokens
    a2_rows.append((str(size), f"{len(chunks_for_size):,}", f"{A2_HIT[size]:.0f}", f"{tokens:.0f}"))
print_table(["Chunk size (tokens)", "Total chunks", "Hit rate@3 (%)", "Tokens sent per query (top 3)"],
            a2_rows, title="A2 chunk size test (top 3, 10 queries)", save_as="a2_chunk_size")

### A2.2 Chosen chunk size (1 mark)

Choose using hit rate@3 and retrieved-token cost, subject to the generator's configured context capacity. Count all ten complete prompts, including the shared instructions, question, chunk labels, and Assignment 1 chat template, and reserve output tokens. The adapter's 512-token training sequence length is a training-distribution consideration, not the model's architectural context limit. Actual reranked prompts are checked again in Part D.


In [ ]:
from transformers import AutoConfig, AutoTokenizer

GEN_MAX_NEW_TOKENS = 120
# Training sequence length is not the model's architectural context capacity.
ADAPTER_TRAINING_LENGTH = 512
# Reuse this exact prompt in both budget measurements and final generation.
GEN_PROMPT = """Answer the question using ONLY the numbered chunks below.
Cite the chunk number you used after each fact, for example [2]. Use only the numbers 1, 2 and 3.
Stop after your answer. Do not write a new question or a new instruction.
If the chunks do not contain the answer, reply exactly: Not found in the documents

Chunks:
[1] {chunk_1}
[2] {chunk_2}
[3] {chunk_3}

Question: {question}"""


def build_generator_prompt(tokenizer, question, chunk_texts):
    if len(chunk_texts) != 3:
        raise ValueError("Generation requires exactly three chunks.")
    content = GEN_PROMPT.format(chunk_1=chunk_texts[0], chunk_2=chunk_texts[1],
                                chunk_3=chunk_texts[2], question=question)
    return tokenizer.apply_chat_template([{"role": "user", "content": content}],
                                         tokenize=False, add_generation_prompt=True).rstrip()


mistral_tok = AutoTokenizer.from_pretrained(ADAPTER_DIR)
generator_config = AutoConfig.from_pretrained(CPT_DIR)
# Ignore effectively unlimited tokenizer sentinels; respect any finite tokenizer limit.
context_limits = [int(generator_config.max_position_embeddings)]
if 0 < mistral_tok.model_max_length < 1_000_000:
    context_limits.append(int(mistral_tok.model_max_length))
GEN_CONTEXT_LIMIT = min(context_limits)
PROMPT_LENGTHS = {}
for size in A2_SIZES:
    retrieved = dense_search(chunk_sets[size], [item["query"] for item in QUERIES], 3)
    # Include instructions, question, chunk labels, and the actual chat wrapper.
    PROMPT_LENGTHS[size] = [len(mistral_tok(build_generator_prompt(mistral_tok, item["query"],
                            [chunk["text"] for chunk in top]), add_special_tokens=False).input_ids)
                            for item, top in zip(QUERIES, retrieved)]
# Every query must fit while leaving room for the generated answer.
feasible_sizes = [size for size in A2_SIZES
                  if max(PROMPT_LENGTHS[size]) + GEN_MAX_NEW_TOKENS <= GEN_CONTEXT_LIMIT]
if not feasible_sizes:
    raise ValueError("Neither chunk size fits all complete prompts plus the answer budget; revise the context policy.")
# Prefer hit rate, then lower retrieved-token cost, then the smaller chunk size.
CHUNK_SIZE = min(feasible_sizes, key=lambda size: (-A2_HIT[size], A2_TOKENS[size], size))
chunks = chunk_sets[CHUNK_SIZE]
print_table(["Chunk size", "Minimum prompt tokens", "Maximum prompt tokens", "Output budget", "Context limit"],
            [(str(size), str(min(PROMPT_LENGTHS[size])), str(max(PROMPT_LENGTHS[size])),
              str(GEN_MAX_NEW_TOKENS), str(GEN_CONTEXT_LIMIT)) for size in A2_SIZES],
            title="Complete dense top-3 prompt budgets for all ten queries", save_as="a2_prompt_budgets")
print(f"Chosen chunk size: {CHUNK_SIZE}; adapter training length: {ADAPTER_TRAINING_LENGTH}")

In [ ]:
from IPython.display import Markdown, display

training_note = ("Some complete prompts exceed the adapter's 512-token training sequence length, so answer quality "
                 "at those lengths must be inspected; this is not an architectural context-limit violation."
                 if max(PROMPT_LENGTHS[CHUNK_SIZE]) > ADAPTER_TRAINING_LENGTH else
                 "All measured prompts are also within the adapter's 512-token training sequence length.")
display(Markdown(
    f"**A2.2: Use {CHUNK_SIZE}-token chunks in Parts B-D.** The 100/400-token configurations achieved "
    f"{A2_HIT[100]:.0f}%/{A2_HIT[400]:.0f}% hit rate@3 with "
    f"{A2_TOKENS[100]:.0f}/{A2_TOKENS[400]:.0f} retrieved tokens per query; among context-feasible sizes, "
    f"we prioritize hit rate and break ties by lower retrieved-token cost. "
    f"The selected configuration's longest complete prompt is {max(PROMPT_LENGTHS[CHUNK_SIZE])} tokens, "
    f"plus {GEN_MAX_NEW_TOKENS} answer tokens, within the configured {GEN_CONTEXT_LIMIT}-token capacity; {training_note}"))


### A1.2 Dense search setup (1 mark)

Reported for the final chunk set chosen in A2.2.


In [ ]:
print_table(["Setting", "Value"], [
    ("Embedding model", EMBED_MODEL_ID),
    ("Vector size", str(EMBED_DIM)),
    ("Similarity measure", "inner product on L2-normalised vectors (cosine)"),
    ("Batch size", str(EMBED_BATCH)),
    ("Number of chunks", f"{len(chunks):,}"),
    ("Index build time (s)", f"{index_build_seconds(chunks):.2f}"),
], title=f"A1.2 dense search setup, chunk size {CHUNK_SIZE} tokens", save_as="a1_2_setup")


## Part B - Search Comparison (4 marks)

Use the chosen chunks and the same ten queries. Compare Dense, BM25, and Hybrid at top five. Hybrid fuses the top 20 from each method using `1 / (60 + rank)` per list; ranks start at one and missing entries contribute zero.

Report mean per-query latency after one warm-up, hit rate@1, and hit rate@3. A hit requires the annotated answer span.


In [ ]:
def evaluate(search_fn, k=5):
    """Run search_fn(query, k) on the 10 queries, after one warm-up run.

    Returns the average latency (ms), hit rate@1 (%), hit rate@3 (%) and the ranked lists for each query.
    """
    # Exclude the warm-up so one-time setup does not inflate measured query latency.
    search_fn(QUERIES[0]["query"], k)
    latencies, ranked = [], {}
    for q in QUERIES:
        t0 = time.perf_counter()
        ranked[q["qid"]] = search_fn(q["query"], k)
        latencies.append((time.perf_counter() - t0) * 1000)

    def hit_at(n):
        """Share of queries whose top n results contain the answer span, in %."""
        # Multiple matching chunks still contribute only one successful query.
        found = sum(any(is_hit(r["text"], q["span"]) for r in ranked[q["qid"]][:n]) for q in QUERIES)
        return 100 * found / len(QUERIES)

    return {"latency_ms": sum(latencies) / len(latencies), "hit1": hit_at(1), "hit3": hit_at(3), "ranked": ranked}
print("evaluate() defined: average latency, hit rate@1 and hit rate@3 for one search method.")

### B1 Dense (1 mark)


In [ ]:
b_results = {}
dense_fn = lambda q, k: dense_search(chunks, [q], k)[0]
b_results["Dense"] = evaluate(dense_fn)
print(f"Dense: hit rate@1 {b_results['Dense']['hit1']:.0f}%, hit rate@3 {b_results['Dense']['hit3']:.0f}%")


### B2 BM25 (1 mark)


In [ ]:
from rank_bm25 import BM25Okapi


def bm25_tokens(text):
    """Split text into lowercase words and numbers."""
    # Keep clinical terms such as HbA1c and SGLT2 together as single tokens.
    return re.findall(r"[a-z0-9]+", text.lower())


# BM25 uses exactly the same chunk collection as dense search.
bm25 = BM25Okapi([bm25_tokens(c["text"]) for c in chunks])


def bm25_fn(query, k):
    """Top k chunks for one query, in the same format as dense_search."""
    scores = bm25.get_scores(bm25_tokens(query))
    # Negating scores makes argsort return the highest-scoring chunks first.
    top = np.argsort(-scores)[:k]
    return [{"rank": r + 1, "chunk_id": chunks[j]["chunk_id"], "score": float(scores[j]), "text": chunks[j]["text"]}
            for r, j in enumerate(top)]


b_results["BM25"] = evaluate(bm25_fn)
print(f"BM25: hit rate@1 {b_results['BM25']['hit1']:.0f}%, hit rate@3 {b_results['BM25']['hit3']:.0f}%")

### B3 Hybrid with RRF (1 mark)


In [ ]:
def rrf_fuse(dense_ids, bm25_ids, k=60, top=5):
    """Reciprocal rank fusion. Each chunk gets the sum of 1 / (k + rank) over the lists it appears in."""
    score = {}
    for ids in (dense_ids, bm25_ids):
        # RRF combines ranks, not incompatible dense and BM25 score scales.
        for rank, cid in enumerate(ids, start=1):
            score[cid] = score.get(cid, 0.0) + 1.0 / (k + rank)
    return sorted(score.items(), key=lambda item: -item[1])[:top]


chunk_by_id = {c["chunk_id"]: c for c in chunks}


def hybrid_fn(query, k):
    """Top 20 from dense and top 20 from BM25, fused with RRF, keep the top k."""
    # Fuse a wider candidate pool so moderately ranked shared results can rise to the top.
    dense_top = dense_fn(query, 20)
    bm25_top = bm25_fn(query, 20)
    fused = rrf_fuse([r["chunk_id"] for r in dense_top], [r["chunk_id"] for r in bm25_top], top=k)
    return [{"rank": r + 1, "chunk_id": cid, "score": s, "text": chunk_by_id[cid]["text"]}
            for r, (cid, s) in enumerate(fused)]


b_results["Hybrid (RRF)"] = evaluate(hybrid_fn)
print(f"Hybrid (RRF): hit rate@1 {b_results['Hybrid (RRF)']['hit1']:.0f}%, "
      f"hit rate@3 {b_results['Hybrid (RRF)']['hit3']:.0f}%")

In [ ]:
def rank_of(method, qid, span):
    """Rank of the first top-5 chunk that contains the span. Returns "-" if none does."""
    for r in b_results[method]["ranked"][qid]:
        if is_hit(r["text"], span):
            return str(r["rank"])
    return "-"


print_table(["Method", "Average query latency (ms)", "Hit rate@1 (%)", "Hit rate@3 (%)"],
            [(m, f"{v['latency_ms']:.1f}", f"{v['hit1']:.0f}", f"{v['hit3']:.0f}") for m, v in b_results.items()],
            title=f"Part B search comparison (chunk size {CHUNK_SIZE}, top 5)", save_as="b_comparison")


### B4 Best method (1 mark)


In [ ]:
BEST_METHOD = max(b_results, key=lambda method: (b_results[method]["hit1"], b_results[method]["hit3"],
                                                -b_results[method]["latency_ms"]))

def answer_rank(method, item):
    value = rank_of(method, item["qid"], item["span"])
    return int(value) if value.isdigit() else 99

example = next((item for item in QUERIES
                if answer_rank(BEST_METHOD, item) < min(answer_rank(method, item)
                   for method in b_results if method != BEST_METHOD)), None)
if example is None:
    example = next((item for item in QUERIES
                    if len({answer_rank(method, item) for method in b_results}) > 1), QUERIES[0])
best = b_results[BEST_METHOD]
tied = [method for method in b_results if method != BEST_METHOD and
        (b_results[method]["hit1"], b_results[method]["hit3"]) == (best["hit1"], best["hit3"])]
choice_reason = (f"Its hit rates tie with {', '.join(tied)}, so lower measured latency breaks the tie."
                 if tied else "It wins by hit rate@1, followed by hit rate@3.")
rank_description = "; ".join(f"{method}: rank {rank_of(method, example['qid'], example['span'])}"
                              for method in b_results)
display(Markdown(
    f"**B4: {BEST_METHOD} is selected**, with {best['hit1']:.0f}% hit rate@1, {best['hit3']:.0f}% "
    f"hit rate@3 and {best['latency_ms']:.1f} ms/query; {choice_reason} "
    f"For {example['qid']} ('{example['query']}'), the annotated answer span ranks as follows: "
    f"{rank_description} ('-' means absent from the top five). "
    "This example shows the observed ranking outcome, not proof of a semantic or keyword mechanism; "
    "latency tie-breaks and conclusions are limited to these ten queries on this hardware."))
example_rows = []
for method, result in b_results.items():
    for chunk in result["ranked"][example["qid"]]:
        example_rows.append((method, str(chunk["rank"]), str(is_hit(chunk["text"], example["span"])),
                             chunk["chunk_id"], chunk["text"]))
show_md_table(["Method", "Rank", "Span hit", "Chunk", "Text"], example_rows,
              title=f"B4 evidence: {example['qid']}; answer span: {example['span']}", save_as="b4_example")


## Part C - Reranking (4 marks)

Rerank the best Part B method's top ten candidates with a cross-encoder and retain three. Compare rankings, hit rate@1, and scoring latency. Rank change rate is the share of queries whose first result changes.

### C1 Rerank (1 mark)


In [ ]:
import torch
from sentence_transformers import CrossEncoder

RERANK_MODEL_ID = "cross-encoder/ms-marco-MiniLM-L-6-v2"
with working("Loading the cross-encoder"):
    ce = CrossEncoder(RERANK_MODEL_ID, device=L.DEVICE)
print(f"cross-encoder: {RERANK_MODEL_ID} on {L.DEVICE}")

best_fn = {"Dense": dense_fn, "BM25": bm25_fn, "Hybrid (RRF)": hybrid_fn}[BEST_METHOD]


def rerank_top3(query):
    """Rerank the best method's top 10 for one query. Returns (top 3 before, top 3 after, seconds spent)."""
    top10 = best_fn(query, 10)
    # The cross-encoder reads each query and candidate chunk together.
    pairs = [(query, c["text"]) for c in top10]
    t0 = time.perf_counter()
    scores = ce.predict(pairs)
    if L.IS_GPU:
        # Wait for GPU scoring to finish before stopping the reranking timer.
        torch.cuda.synchronize()
    seconds = time.perf_counter() - t0
    order = sorted(range(len(top10)), key=lambda i: -scores[i])[:3]
    after = [dict(top10[i], rank=r + 1, ce_score=float(scores[i])) for r, i in enumerate(order)]
    return top10[:3], after, seconds


# Exclude one warm-up; preserve both rankings for the per-query comparison.
ce.predict([(QUERIES[0]["query"], "warm-up")])
final_chunks, before_top3, rerank_ms = {}, {}, []
for q in QUERIES:
    before, after, seconds = rerank_top3(q["query"])
    before_top3[q["qid"]] = before
    final_chunks[q["qid"]] = after
    rerank_ms.append(seconds * 1000)
print(f"reranked {len(QUERIES)} queries; kept the top 3 of 10 for each")

### C2 Compare rankings (1 mark)


In [ ]:
changed = 0
comparison_rows = []
for item in QUERIES:
    qid = item["qid"]
    for stage, ranking in (("Before", before_top3[qid]), ("After", final_chunks[qid])):
        for chunk in ranking:
            comparison_rows.append((qid, item["query"], stage, str(chunk["rank"]), chunk["chunk_id"],
                                    str(is_hit(chunk["text"], item["span"])), chunk["text"]))
    changed += before_top3[qid][0]["chunk_id"] != final_chunks[qid][0]["chunk_id"]
RANK_CHANGE = 100 * changed / len(QUERIES)
show_md_table(["Query", "Question", "Stage", "Rank", "Chunk", "Span hit", "Full chunk text"],
              comparison_rows, title="C2 top three before and after reranking", save_as="c2_rankings")
print(f"Rank change rate: {RANK_CHANGE:.0f}% ({changed}/{len(QUERIES)} first results changed)")


### C3 Measure (1 mark)


In [ ]:
hit1_before = b_results[BEST_METHOD]["hit1"]
hit1_after = 100 * sum(is_hit(final_chunks[q["qid"]][0]["text"], q["span"]) for q in QUERIES) / len(QUERIES)
RERANK_MS = sum(rerank_ms) / len(rerank_ms)
print_table(["Measure", "Value"], [
    ("Reranking time per query (ms)", f"{RERANK_MS:.1f}"),
    ("Rank change rate (%)", f"{RANK_CHANGE:.0f}"),
    ("Hit rate@1 before reranking (from Part B, same method)", f"{hit1_before:.0f}"),
    ("Hit rate@1 after reranking", f"{hit1_after:.0f}"),
], title="Part C reranking measures", save_as="c_measures")


### C4 Did reranking help? (1 mark)


In [ ]:
improved, worsened = [], []
for item in QUERIES:
    qid = item["qid"]
    before_hit = is_hit(before_top3[qid][0]["text"], item["span"])
    after_hit = is_hit(final_chunks[qid][0]["text"], item["span"])
    if after_hit and not before_hit:
        improved.append(qid)
    elif before_hit and not after_hit:
        worsened.append(qid)
verdict = ("improved top-1 span retrieval" if hit1_after > hit1_before else
           "left aggregate top-1 span retrieval unchanged" if hit1_after == hit1_before else
           "reduced top-1 span retrieval accuracy")
display(Markdown(
    f"**C4:** Reranking {verdict}: hit rate@1 changed from {hit1_before:.0f}% to {hit1_after:.0f}% "
    f"at an additional {RERANK_MS:.1f} ms/query for cross-encoder scoring. "
    f"It gained top-1 hits for {', '.join(improved) or 'no queries'} and lost them for "
    f"{', '.join(worsened) or 'no queries'}, as the full C2 rankings show. "
    f"On this ten-query benchmark, the measured trade-off is {hit1_after-hit1_before:+.0f} percentage points "
    "of hit rate@1 for that latency cost; these measurements alone do not establish the reason for any domain mismatch."))


## Part D - Generate and Evaluate Answers (7 marks)

### D1 Grounded Generation (3 marks)

Use the Assignment 1 generator with the three numbered, reranked chunks. Show the grounding prompt, all ten cited answers, and one out-of-corpus refusal.


In [ ]:
import torch
import transformers
from packaging.version import Version
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

GEN_BASE_ID = "mistralai/Mistral-7B-v0.1"
GENERATOR_NAME = f"{GEN_BASE_ID} + CPT + QLoRA adapter (Assignment 1)"
GEN_STOPS = ["Instruction:"]
# Transformers renamed the loading keyword; support both installed API versions.
DTYPE_KW = "dtype" if Version(transformers.__version__) >= Version("4.56") else "torch_dtype"
if not L.IS_GPU:
    raise RuntimeError("NF4 model loading requires the lab CUDA GPU.")
COMPUTE_DTYPE = getattr(torch, L.DTYPE) if L.DTYPE in ("bfloat16", "float16") else torch.float16
# NF4 controls weight storage; the compute dtype is selected separately for this GPU.
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                bnb_4bit_compute_dtype=COMPUTE_DTYPE, bnb_4bit_use_double_quant=True)
with working("Loading the CPT generator and QLoRA adapter", folder=CPT_DIR):
    # Restore the continued-pretraining checkpoint before attaching its trained adapter.
    gen_base = AutoModelForCausalLM.from_pretrained(
        CPT_DIR, quantization_config=bnb_config, **{DTYPE_KW: COMPUTE_DTYPE}, device_map={"": 0})
    gen_model = PeftModel.from_pretrained(gen_base, ADAPTER_DIR).eval()
gen_tok = AutoTokenizer.from_pretrained(ADAPTER_DIR)
print(f"Generator: {GENERATOR_NAME}; NF4 compute dtype: {COMPUTE_DTYPE}")
print(f"Shared answer budget: {GEN_MAX_NEW_TOKENS}; configured context capacity: {GEN_CONTEXT_LIMIT}")

#### D1.1 Generation prompt (1 mark)


In [ ]:
print(GEN_PROMPT)
print("The same template and chat wrapper were used for the Part A token-budget checks.")


In [ ]:
NOT_FOUND = "Not found in the documents"
CITE = re.compile(r"\[(\d+)\]")


def clean_answer(raw):
    # Remove common response delimiters and continuation into another instruction.
    text = raw
    for stop in GEN_STOPS:
        text = text.split(stop, 1)[0]
    if "Response:" in text:
        text = text.split("Response:", 1)[1]
    return text.strip()


def generate_answer(question, chunk_texts):
    prompt = build_generator_prompt(gen_tok, question, chunk_texts)
    # Never silently truncate evidence to make a prompt fit.
    encoded = gen_tok(prompt, return_tensors="pt", add_special_tokens=False, truncation=False)
    prompt_tokens = encoded["input_ids"].shape[1]
    if prompt_tokens + GEN_MAX_NEW_TOKENS > GEN_CONTEXT_LIMIT:
        raise ValueError(f"Complete prompt ({prompt_tokens}) plus answer budget ({GEN_MAX_NEW_TOKENS}) "
                         f"exceeds context capacity ({GEN_CONTEXT_LIMIT}); no chunks were truncated.")
    ids = encoded.to(gen_model.device)
    with torch.no_grad():
        generated = gen_model.generate(**ids, max_new_tokens=GEN_MAX_NEW_TOKENS, do_sample=False,
                                        pad_token_id=gen_tok.eos_token_id)
    # Decode only new tokens, not the question and retrieved chunks in the input.
    raw = gen_tok.decode(generated[0][prompt_tokens:], skip_special_tokens=True)
    return clean_answer(raw)


def is_not_found(answer):
    return answer.strip().rstrip(".").strip() == NOT_FOUND


def has_valid_citation(answer):
    # Valid citation numbers establish format, not whether the cited evidence supports a claim.
    numbers = [int(number) for number in CITE.findall(answer)]
    return bool(numbers) and all(1 <= number <= 3 for number in numbers)


# Reranking changes the chosen chunks, so check the actual Part D prompts again.
actual_budgets = []
for item in QUERIES:
    prompt = build_generator_prompt(gen_tok, item["query"],
                                    [chunk["text"] for chunk in final_chunks[item["qid"]]])
    length = len(gen_tok(prompt, add_special_tokens=False).input_ids)
    actual_budgets.append((item["qid"], str(length), str(GEN_MAX_NEW_TOKENS)))
    if length + GEN_MAX_NEW_TOKENS > GEN_CONTEXT_LIMIT:
        raise ValueError(f"{item['qid']}: reranked prompt exceeds context capacity.")
print_table(["Query", "Complete reranked prompt tokens", "Answer token budget"], actual_budgets,
            title="Part D actual prompt budgets", save_as="d1_prompt_budgets")

#### D1.2 All 10 answers (1 mark)


In [ ]:
answers = {}
for item in QUERIES:
    qid = item["qid"]
    texts = [chunk["text"] for chunk in final_chunks[qid]]
    answers[qid] = generate_answer(item["query"], texts)
    cited = ",".join(sorted(set(CITE.findall(answers[qid])))) or "-"
    print(f"\n{qid}: {item['query']}\nAnswer: {answers[qid]}\nChunks cited: {cited}")


#### D1.3 A query whose answer is not in the corpus (1 mark)


In [ ]:
OOC_QUERY = "Who won the 2018 FIFA World Cup?"
_, ooc_top3, _ = rerank_top3(OOC_QUERY)
for i, c in enumerate(ooc_top3, 1):
    print(f"[{i}] {c['chunk_id']}: {c['text'][:120]}")
ooc_answer = generate_answer(OOC_QUERY, [c["text"] for c in ooc_top3])
print(f"\nQuestion: {OOC_QUERY}")
print(f"Reply: {ooc_answer}")
print(f"Replied Not found: {'yes' if is_not_found(ooc_answer) else 'no'}")


#### D1 switch to an instruct model (not used in this notebook)

This notebook answers all 10 queries and the out-of-corpus query with the Assignment 1 model only, so no switch is made.

### D2 Check the answers with an LLM judge (4 marks)


In [ ]:
import gc

# Retrieval and generation are complete; release GPU memory before loading the judge.
embed_model.to("cpu")
ce.model.to("cpu")
gen_model = None
gen_base = None
gc.collect()
torch.cuda.empty_cache()

# Use a different model family to avoid asking the generator to grade its own answers.
JUDGE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
assert JUDGE_MODEL_ID != GEN_BASE_ID, "The judge must differ from the generator."
JUDGE_MAX_NEW_TOKENS = 256
with working("Loading the judge in NF4 with an explicit compute dtype"):
    judge_tok = AutoTokenizer.from_pretrained(JUDGE_MODEL_ID)
    judge_model = AutoModelForCausalLM.from_pretrained(
        JUDGE_MODEL_ID, quantization_config=bnb_config,
        **{DTYPE_KW: COMPUTE_DTYPE}, device_map={"": 0}).eval()
# Sampling options are irrelevant because evaluation uses greedy decoding.
judge_model.generation_config.temperature = None
judge_model.generation_config.top_p = None
judge_model.generation_config.top_k = None
JUDGE_CONTEXT_LIMIT = int(judge_model.config.max_position_embeddings)
if 0 < judge_tok.model_max_length < 1_000_000:
    JUDGE_CONTEXT_LIMIT = min(JUDGE_CONTEXT_LIMIT, int(judge_tok.model_max_length))


def judge_generate_raw(prompt_text):
    text = judge_tok.apply_chat_template([{"role": "user", "content": prompt_text}],
                                         tokenize=False, add_generation_prompt=True)
    ids = judge_tok(text, return_tensors="pt", add_special_tokens=False, truncation=False)
    # Keep the complete reference, answer, and evidence while reserving reply tokens.
    if ids["input_ids"].shape[1] + JUDGE_MAX_NEW_TOKENS > JUDGE_CONTEXT_LIMIT:
        raise ValueError("Judge prompt plus reply budget exceeds context capacity; no evidence was truncated.")
    ids = ids.to(judge_model.device)
    with torch.no_grad():
        generated = judge_model.generate(**ids, max_new_tokens=JUDGE_MAX_NEW_TOKENS,
                                          do_sample=False, pad_token_id=judge_tok.eos_token_id)
    return judge_tok.decode(generated[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)


print(f"Judge: {JUDGE_MODEL_ID}; NF4 storage; compute dtype: {COMPUTE_DTYPE}")
print(f"Judge context limit: {JUDGE_CONTEXT_LIMIT}; CUDA allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GiB")

#### D2.1 Judge prompt and results (1 mark)


In [ ]:
JUDGE_PROMPT = """You are grading one answer about Type 2 Diabetes.

Question: {question}
Expected answer (reference): {expected}
Retrieved chunks:
[1] {chunk_1}
[2] {chunk_2}
[3] {chunk_3}
Model answer: {answer}

Check 1, CORRECT: compare the model answer with the expected answer.
  CORRECT = it states the same key fact as the expected answer.
  PARTLY CORRECT = it states only part of the key fact, or the right fact with a detail missing or wrong.
  WRONG = it states a different fact, contradicts the expected answer, or says "Not found in the documents".
Check 2, SUPPORTED: can every statement in the model answer be found in the 3 chunks above?
  YES = every statement is in the chunks. NO = at least one statement is not.
  A "Not found in the documents" reply makes no claims, so it is SUPPORTED: YES.
  A reply of exactly "Not found in the documents" always gets CORRECT: WRONG and SUPPORTED: YES.
REASON: one sentence. If SUPPORTED is NO, the reason must quote the unsupported statement in double quotes.

Reply in exactly this format, with nothing else:
CORRECT: <CORRECT | PARTLY CORRECT | WRONG>
SUPPORTED: <YES | NO>
REASON: <one sentence>"""
print(JUDGE_PROMPT)


In [ ]:
def parse_verdict(text):
    # Tolerate simple Markdown decoration, but reject missing or duplicate label lines.
    cleaned = re.sub(r"[<>*]", "", text)
    fields = {}
    for label in ("CORRECT", "SUPPORTED", "REASON"):
        matches = re.findall(rf"^\s*{label}\s*:\s*([^\n]+)", cleaned, flags=re.I | re.M)
        fields[label] = matches[0].strip() if len(matches) == 1 else ""
    correct = fields["CORRECT"].rstrip(".").upper()
    supported = fields["SUPPORTED"].rstrip(".").upper()
    return {"correct": correct if correct in {"CORRECT", "PARTLY CORRECT", "WRONG"} else "UNPARSED",
            "supported": supported if supported in {"YES", "NO"} else "UNPARSED",
            "reason": fields["REASON"]}


def valid_verdict(verdict):
    # Unsupported verdicts must quote the offending statement, not just label it NO.
    return (verdict["correct"] in {"CORRECT", "PARTLY CORRECT", "WRONG"}
            and verdict["supported"] in {"YES", "NO"}
            and bool(verdict["reason"].strip())
            and (verdict["supported"] != "NO" or bool(re.search(r'"[^"\n]+"', verdict["reason"]))))


JUDGE_ATTEMPTS = []


def judge_generate(prompt_text):
    retry_prompt = prompt_text
    for attempt in range(3):
        raw = judge_generate_raw(retry_prompt)
        parsed = parse_verdict(raw)
        # Preserve every raw attempt so formatting failures remain visible in the audit.
        JUDGE_ATTEMPTS.append({"attempt": attempt + 1, "prompt": retry_prompt, "raw": raw,
                               "valid": valid_verdict(parsed)})
        (OUT_DIR / "judge_attempts.json").write_text(
            json.dumps(JUDGE_ATTEMPTS, indent=2, ensure_ascii=False), encoding="utf-8")
        if valid_verdict(parsed):
            return raw
        # Retry against the original evidence; do not silently invent missing labels.
        retry_prompt = (prompt_text + "\n\nYour previous response did not meet the required format:\n" + raw
                        + '\nRe-evaluate the original evidence and return exactly three labeled lines: '
                        'CORRECT, SUPPORTED, REASON. Use only the allowed labels, include a nonempty reason, '
                        'and quote the unsupported statement in double quotes when SUPPORTED is NO.')
    raise ValueError("Judge response remains invalid after three attempts; inspect outputs/judge_attempts.json.")


verdicts = {}
print("Judge replies are validated, retried at most twice, and preserved for audit.")

In [ ]:
verdicts = {}
for item in QUERIES:
    qid = item["qid"]
    texts = [chunk["text"] for chunk in final_chunks[qid]]
    prompt = JUDGE_PROMPT.format(question=item["query"], expected=item["expected_answer"],
                                 chunk_1=texts[0], chunk_2=texts[1], chunk_3=texts[2], answer=answers[qid])
    raw = judge_generate(prompt)
    verdicts[qid] = dict(parse_verdict(raw), raw=raw)
    verdict = verdicts[qid]
    print(f"{qid}: {verdict['correct']} / {verdict['supported']} - {verdict['reason']}")


In [ ]:
expected_ids = {item["qid"] for item in QUERIES}
assert len(expected_ids) == 10 and set(verdicts) == expected_ids, "All ten queries must have judge verdicts."
assert set(answers) == expected_ids, "All ten generated answers are required."
assert all(valid_verdict(value) for value in verdicts.values()), "Resolve invalid judge responses before counting."
show_md_table(["Query", "Correct", "Supported", "Reason"],
              [(f"{item['qid']}: {item['query']}", verdicts[item["qid"]]["correct"],
                verdicts[item["qid"]]["supported"], verdicts[item["qid"]]["reason"]) for item in QUERIES],
              title="D2.1 judge results", save_as="d2_verdicts")
counts = {label: sum(value["correct"] == label for value in verdicts.values())
          for label in ("CORRECT", "PARTLY CORRECT", "WRONG")}
assert sum(counts.values()) == 10
print_table(["Check", "Count out of 10"], [
    ("Correct", str(counts["CORRECT"])), ("Partly correct", str(counts["PARTLY CORRECT"])),
    ("Wrong", str(counts["WRONG"])),
    ("Supported by the chunks", str(sum(value["supported"] == "YES" for value in verdicts.values())))],
    title="D2.1 complete summary", save_as="d2_summary")
(OUT_DIR / "answers_and_verdicts.json").write_text(
    json.dumps({"queries": QUERIES, "chunks": final_chunks, "answers": answers, "verdicts": verdicts,
                "ooc_query": OOC_QUERY, "ooc_answer": ooc_answer}, indent=2, ensure_ascii=False), encoding="utf-8")
show_md_table(["Query", "Not found", "Citation format valid"],
              [(item["qid"], str(is_not_found(answers[item["qid"]])),
                "not applicable" if is_not_found(answers[item["qid"]]) else
                str(has_valid_citation(answers[item["qid"]]))) for item in QUERIES],
              title="D1 format audit (citation numbers alone do not prove support)", save_as="d1_format_audit")


#### D2.2 Spot-check the judge (1 mark)

*The cell below shows 3 answers (with different verdicts where possible), their 3 chunks and the judge's verdicts. Read them, then fill in the notes cell after it.*


In [ ]:
SPOT_QIDS = []
seen_verdicts = set()
for item in QUERIES:
    value = verdicts[item["qid"]]
    signature = (value["correct"], value["supported"])
    if signature not in seen_verdicts and len(SPOT_QIDS) < 3:
        SPOT_QIDS.append(item["qid"])
        seen_verdicts.add(signature)
for item in QUERIES:
    if len(SPOT_QIDS) == 3:
        break
    if item["qid"] not in SPOT_QIDS:
        SPOT_QIDS.append(item["qid"])
for qid in SPOT_QIDS:
    item = next(value for value in QUERIES if value["qid"] == qid)
    print(f"\n{qid}: {item['query']}\nExpected: {item['expected_answer']}\nAnswer: {answers[qid]}")
    for number, chunk in enumerate(final_chunks[qid], 1):
        print(f"\n[{number}] {chunk['chunk_id']}\n{chunk['text']}")
    value = verdicts[qid]
    print(f"\nJudge: {value['correct']} / {value['supported']}; {value['reason']}")


**D2.2 Human verdicts.** After reading the three cases above, enter the decisions in `SPOT_REVIEW` in the next cell, keyed by the displayed query numbers. Each entry needs `correct` (`CORRECT`, `PARTLY CORRECT`, or `WRONG`), `supported` (`YES` or `NO`), and `reason` explaining your own comparison with the expected answer and chunks. If you disagree with either judge label, quote the relevant answer/chunk evidence in your reason. Agreement counts are computed from your decisions; they are not inferred from the judge itself.


In [ ]:
# Enter independent human judgments after reading each answer and its three chunks.
SPOT_REVIEW = {}

spot_rows, spot_pending, spot_disagreements = [], [], []
spot_agreements = 0
for qid in SPOT_QIDS:
    mine = SPOT_REVIEW.get(qid, {})
    # Incomplete reviews stay pending instead of being inferred from the judge's labels.
    if (mine.get("correct") not in {"CORRECT", "PARTLY CORRECT", "WRONG"}
            or mine.get("supported") not in {"YES", "NO"}
            or not str(mine.get("reason", "")).strip()):
        spot_pending.append(qid)
        continue
    judge = verdicts[qid]
    # Agreement requires matching both correctness and evidence-support assessments.
    agrees = (mine["correct"], mine["supported"]) == (judge["correct"], judge["supported"])
    spot_agreements += agrees
    spot_rows.append((qid, f"{judge['correct']} / {judge['supported']}",
                      f"{mine['correct']} / {mine['supported']}", str(agrees), mine["reason"]))
    if not agrees:
        spot_disagreements.append(f"{qid}: {mine['reason']}")
if spot_rows:
    show_md_table(["Query", "Judge", "Human", "Agree on both checks", "Human explanation"], spot_rows,
                  title="D2.2 human spot-check", save_as="d2_spot_check")
if spot_pending:
    print(f"Human review pending: {', '.join(spot_pending)}. Fill SPOT_REVIEW and rerun this cell.")
else:
    display(Markdown(f"I agree with **{spot_agreements} of 3** complete verdicts. " +
                     ("Disagreements: " + " ".join(spot_disagreements) if spot_disagreements else
                      "There are no disagreements; the table records my evidence-based reasons.")))

#### D2.3 Find the cause (1 mark)

Inspect one genuinely wrong or unsupported answer and its three full chunks. The literal span check is only a diagnostic: equivalent information may be present without containing the annotated wording. Determine whether the needed facts were available, identify the specific answer error, and attribute it to retrieval, generation, or both. If every answer genuinely passes, report that result honestly and ask the evaluator what alternative analysis is acceptable; do not invent a failed answer.


In [ ]:
FAILED_QUERY_OVERRIDE = ""


def failed(qid):
    judge = verdicts[qid]
    human = SPOT_REVIEW.get(qid, {})
    return (judge["correct"] == "WRONG" or judge["supported"] == "NO"
            or human.get("correct") == "WRONG" or human.get("supported") == "NO")


if FAILED_QUERY_OVERRIDE:
    if FAILED_QUERY_OVERRIDE not in verdicts or not failed(FAILED_QUERY_OVERRIDE):
        raise ValueError("Choose a genuinely wrong or unsupported answer, with a recorded judge or human verdict.")
    FAILED_QID = FAILED_QUERY_OVERRIDE
else:
    FAILED_QID = next((item["qid"] for item in QUERIES if failed(item["qid"])), None)
if FAILED_QID is None:
    print("No wrong or unsupported answer has been identified. Recheck the evidence; do not manufacture a failure. "
          "If all ten genuinely pass, report that limitation and confirm the D2.3 alternative with the evaluator.")
else:
    item = next(value for value in QUERIES if value["qid"] == FAILED_QID)
    print(f"{FAILED_QID}: {item['query']}\nExpected: {item['expected_answer']}\nAnswer: {answers[FAILED_QID]}")
    print(f"Judge verdict: {verdicts[FAILED_QID]}")
    for number, chunk in enumerate(final_chunks[FAILED_QID], 1):
        print(f"\n[{number}] {chunk['chunk_id']}\n{chunk['text']}")
    literal_hit = any(is_hit(chunk["text"], item["span"]) for chunk in final_chunks[FAILED_QID])
    print(f"\nAnnotated span: {item['span']}\nLiteral span present: {literal_hit}")
    print("This is a lexical diagnostic only. Check whether equivalent information occurs anywhere in the three "
          "chunks before deciding whether retrieval, generation, or both caused the failure.")


**Human cause analysis.** Complete `FAILURE_REVIEW` in the final code cell after inspecting the selected answer above. Record `qid`, `needed_information_present` (a Python Boolean), `cause` (`retrieval`, `generation`, or `both`), `evidence` (a relevant chunk/answer quote or an explanation of which fact is missing), and `explanation` (what went wrong). An unsupported extra claim can be a generation failure even if retrieval also missed another fact. Judge verdicts are evidence to inspect, not unquestionable ground truth.

#### D2.4 Suggest a fix (1 mark)

In `FAILURE_REVIEW`, add `fix` and `why_it_helps`: one concrete change matched to the D2.3 evidence and an explanation of how it addresses that failure. For example, a boundary-split answer may motivate a different allowed chunk size; a low-ranked answer-bearing candidate may motivate a retrieval or reranking change; ignoring available facts may motivate clearer prompt wording. Do not claim that a proposed fix improved results unless you actually test it.


In [ ]:
FAILURE_REVIEW = {
    "qid": "",
    "needed_information_present": None,
    "cause": "",
    "evidence": "",
    "explanation": "",
    "fix": "",
    "why_it_helps": "",
}

failure_review_complete = (
    FAILED_QID is not None and FAILURE_REVIEW["qid"] == FAILED_QID
    and type(FAILURE_REVIEW["needed_information_present"]) is bool
    and FAILURE_REVIEW["cause"] in {"retrieval", "generation", "both"}
    and all(isinstance(FAILURE_REVIEW[field], str) and FAILURE_REVIEW[field].strip()
            for field in ("evidence", "explanation", "fix", "why_it_helps")))
if failure_review_complete:
    availability = "was present" if FAILURE_REVIEW["needed_information_present"] else "was not fully present"
    display(Markdown(f"**D2.3 ({FAILED_QID}):** The needed information {availability} in the retrieved chunks. "
                     f"Cause: **{FAILURE_REVIEW['cause']}**. {FAILURE_REVIEW['explanation']} "
                     f"Evidence: {FAILURE_REVIEW['evidence']}"))
    display(Markdown(f"**D2.4 proposed fix:** {FAILURE_REVIEW['fix']} "
                     f"Why it should help: {FAILURE_REVIEW['why_it_helps']}"))
else:
    print("D2.3/D2.4 require your evidence-based FAILURE_REVIEW after the lab run.")

completion_checks = {
    "Ten reviewed evaluation query/span pairs": len(QUERIES) == 10 and
        all(item.get("human_reviewed") is True for item in QUERIES),
    "Ten valid judge verdicts": set(verdicts) == {item["qid"] for item in QUERIES} and
        all(valid_verdict(value) for value in verdicts.values()),
    "Three completed human spot-checks": len(SPOT_QIDS) == 3 and not spot_pending,
    "Evidence-based failure analysis and proposed fix": failure_review_complete,
    "Out-of-corpus refusal demonstrated": is_not_found(ooc_answer),
    "All non-refusal answers have valid citation numbers": all(
        is_not_found(answer) or has_valid_citation(answer) for answer in answers.values()),
}
print_table(["Completion check", "Status"],
            [(label, "PASS" if complete else "PENDING") for label, complete in completion_checks.items()],
            title="Notebook content checks", save_as="completion_checks")
(OUT_DIR / "human_review.json").write_text(
    json.dumps({"spot_checks": SPOT_REVIEW, "failure_review": FAILURE_REVIEW,
                "completion_checks": completion_checks}, indent=2, ensure_ascii=False), encoding="utf-8")
print("Citation-format checks do not establish factual support; inspect the judge and human evidence.")
if not completion_checks["Out-of-corpus refusal demonstrated"]:
    print("D1.3 remains unmet. Inspect the Assignment 1 failure; the PDF permits a named instruct-model fallback "
          "with a displayed failure example. If switching, regenerate and rejudge all ten answers, "
          "and keep the judge different from the generator.")
print("Before submission: complete pending reviews, retain all outputs, and export this notebook to HTML.")
